[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/bloc3_ml/cours/seance1_cours.ipynb)

# Séance 3.1 — Décrire, relier, comparer — les statistiques dont le ML a besoin

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- choisir entre moyenne et médiane selon la forme de la distribution, et lire un `describe()`
- mesurer la dispersion et la concentration d'une variable
- mesurer un lien entre deux variables, et reconnaître les trois cas où le coefficient ment
- dire si un écart entre deux groupes peut venir du hasard, et lire une p-value
- compter les individus avant de commenter un résultat

## Une phrase de rapport, et le problème

> *« Le panier moyen de nos clients est de 590 €. »*

Cette phrase se trouve dans à peu près tous les rapports d'activité. Elle a
l'air d'une information. Posez-vous la question suivante :

**Si vous deviez fixer le seuil de livraison gratuite, le mettriez-vous à
590 € ?**

À la fin de cette séance vous saurez pourquoi la réponse est non, et ce qu'il
fallait regarder à la place.

### Où cette séance se situe

Elle ouvre le bloc **machine learning**, et pourtant elle n'en fait pas. Elle
installe les quatre réflexes sans lesquels un modèle prédictif ne veut rien
dire : savoir décrire une variable, savoir mesurer un lien, savoir si un écart
est réel, et savoir compter les individus. Les trois séances suivantes s'en
servent en permanence.

### Les données

Même détaillant que le bloc 2, mais à une **maille** différente : une ligne
n'est plus un produit vendu, c'est **une commande entière**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/Intelligence-Artificielle-et-Data-Science/main/bloc3_ml/data/"

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")   ## une ligne = une COMMANDE

print(cmd.shape)   ## 1 955 commandes, et non 45 123 lignes de vente
cmd.head(3)

Trois grandeurs à ne pas confondre :

| Colonne | Ce que c'est |
|---|---|
| `ca` | le montant total de la commande, en euros |
| `nart` | le nombre de **produits distincts** qu'elle contient |
| `qte` | le nombre total d'**unités** commandées, toutes lignes confondues |

Une commande de 3 produits distincts en 100 exemplaires chacun a donc
`nart` = 3 et `qte` = 300.

## 1. Décrire : la moyenne n'est pas le milieu

Deux façons de résumer une variable par un seul nombre :

- la **moyenne** : on additionne les valeurs sur toutes les observations, puis
  on divise par le **nombre d'observations**.
- la **médiane** : on classe les observations par valeur croissante et on
  retient celle qui occupe le **milieu du classement**. La moitié est en
  dessous, l'autre moitié au-dessus.

In [ ]:
# somme des montants / nombre de commandes
print("moyenne :", round(cmd["ca"].mean(), 2))
# le montant qui coupe les 1 955 commandes classees en deux moities
print("mediane :", round(cmd["ca"].median(), 2))

**590 € contre 356 €.** Un écart de 234 €, soit deux tiers de la médiane. Ces
deux nombres décrivent le même fichier.

Question avant d'exécuter la cellule suivante : à votre avis, quelle
proportion des commandes dépasse la « moyenne » ?

In [ ]:
# (colonne > valeur) donne des True/False ; leur moyenne est une proportion
part = 100 * (cmd["ca"] > cmd["ca"].mean()).mean()

print(round(part, 1), "% des commandes depassent la moyenne")

**29 %.** Sept commandes sur dix sont en dessous de la « moyenne ». Un seuil
de livraison gratuite à 590 € serait hors de portée pour 71 % des commandes.

Regardons pourquoi.

In [ ]:
cmd["ca"].plot(kind="hist", bins=40, figsize=(7, 4))   ## la FORME
plt.title("Repartition de ca")
plt.xlabel("ca : montant de la commande (euros)")
plt.show()

Voilà la forme : un **tassement à gauche** et une **traîne qui s'étire loin à
droite**. Quelques commandes énormes (jusqu'à 16 775 €) tirent la moyenne vers
le haut sans déplacer la médiane d'un centime.

> 💡 **La règle.** Distribution symétrique : moyenne et médiane coïncident,
> prenez l'une ou l'autre. Distribution asymétrique — et **presque tout ce qui
> est en euros l'est** — la médiane décrit la situation typique, la moyenne
> décrit le total divisé par l'effectif. Affichez les deux.

### Tout voir d'un coup : `describe()`

In [ ]:
cmd["ca"].describe().round(2)   ## huit nombres, une distribution

| Ligne | Ce que ça dit ici |
|---|---|
| `count` | 1 955 commandes |
| `mean` | 589,73 € — la moyenne |
| `std` | 918,41 € — l'écart-type, voir plus bas |
| `min` | 1,45 € — la plus petite commande |
| `25%` | 189,65 € — un quart des commandes sont en dessous |
| `50%` | 355,89 € — la médiane |
| `75%` | 659,52 € — trois quarts sont en dessous |
| `max` | 16 774,72 € — la plus grosse |

### Les quantiles répondent aux questions de seuil

In [ ]:
# quantile(0.9) : 90 % des commandes sont EN DESSOUS de ce montant
print("seuil des 10 % du haut :", round(cmd["ca"].quantile(0.9), 2))
print("seuil des 10 % du bas  :", round(cmd["ca"].quantile(0.1), 2))

> ⚠️ **Le piège des quantiles :** ils s'expriment en **proportion**, entre 0
> et 1, jamais en pourcentage. La cellule suivante est volontairement fausse.

In [ ]:
cmd["ca"].quantile(90)   ## erreur volontaire : 90 au lieu de 0.9

Dernière ligne :

```
ValueError: percentiles should all be in the interval [0, 1]
```

Les 10 % du haut, c'est `quantile(0.9)`, pas `quantile(90)`. **Seule la
dernière ligne d'une erreur compte.**

### Découper en tranches — `pd.cut`

Un seuil coupe en deux. Souvent on veut plus : classer chaque commande dans
une **gamme**. `pd.cut` transforme une variable quantitative en variable
qualitative, à partir de bornes qu'on choisit.

In [ ]:
# 5 bornes -> 4 tranches. Les etiquettes sont dans l'ordre des bornes.
cmd["gamme"] = pd.cut(cmd["ca"], bins=[0, 200, 500, 1000, 20000],
                      labels=["petite", "moyenne", "grande", "tres_grande"])

cmd["gamme"].value_counts().sort_index()

Les bornes sont un **choix**, pas une propriété des données : `[0, 200, 500,
1000, 20000]` dit qu'on considère qu'une commande devient « grande » à 500 €.
Un autre découpage donnerait un autre tableau. Annoncez toujours vos bornes en
même temps que vos résultats.

> 💡 Vous retrouverez `pd.cut` en séance 3.4, pour découper l'ancienneté des
> abonnés en tranches.

### Mesurer la dispersion

Deux commandes à 350 € et 360 €, ou deux commandes à 10 € et 700 € : même
moyenne, situation très différente.

In [ ]:
q1 = cmd["ca"].quantile(0.25)   ## un quart des commandes en dessous
q3 = cmd["ca"].quantile(0.75)   ## trois quarts en dessous

print("ecart-type          :", round(cmd["ca"].std(), 2))   ## fragile
print("ecart interquartile :", round(q3 - q1, 2))           ## robuste

L'**écart-type** (`std`) mesure l'écart typique des observations à leur
moyenne, dans la même unité — ici des euros.

**Regardez-le bien : 918 €, soit davantage que la moyenne elle-même.** Sur une
variable positive comme un montant, c'est un **indicateur** de dispersion très
forte. C'est un signe, pas une preuve : 590 € reste la moyenne des 1 955
commandes et garde son sens. Ce que l'indicateur suggère, c'est qu'à ce niveau
de dispersion la moyenne renseigne mal sur ce que dépense **une** commande
prise au hasard. La bonne réaction n'est pas d'y renoncer, c'est de ne pas la
citer seule.

L'**écart interquartile** (q3 − q1, ici 470 €) est sa version robuste : il
décrit la moitié centrale et ignore les extrêmes. Il ne bouge pas si la plus
grosse commande double.

### Où est concentré le chiffre d'affaires ?

In [ ]:
top = cmd["ca"].sort_values(ascending=False)   ## les plus grosses d'abord
n10 = int(0.10 * len(cmd))   ## les 10 % de commandes les plus grosses

part = 100 * top.head(n10).sum() / top.sum()   ## leur poids dans le total
print(n10, "commandes font", round(part, 1), "% du chiffre d'affaires")

**195 commandes sur 1 955 font 41,3 % du chiffre d'affaires.**

Vous avez déjà croisé ce phénomène en séance 2.2 : deux clients irlandais
pesaient 22,7 % du CA. Ce n'était pas une anomalie isolée, c'est la façon dont
ce marché est fait.

Une moyenne **ne contient pas** cette information. Ce n'est pas qu'elle
l'effacerait : elle répond à une autre question — le total rapporté à
l'effectif — et la répartition n'entre nulle part dans ce calcul. La
concentration est une question distincte, elle demande un chiffre distinct.

## 2. Relier deux variables

**On trace d'abord.** Toujours.

In [ ]:
# On trace AVANT de calculer : un coefficient ne montre pas la forme
cmd.plot(kind="scatter", x="qte", y="ca", alpha=0.3, figsize=(7, 4))
plt.title("ca en fonction de qte")
plt.xlabel("qte : nombre d'unites commandees")
plt.ylabel("ca : montant de la commande (euros)")
plt.show()

Le lien saute aux yeux : plus d'unités commandées, plus d'euros. La
**corrélation** met un nombre sur cette impression. Elle vaut **1** pour une
droite croissante parfaite, **−1** pour une droite décroissante parfaite, et
**0** quand il n'y a aucune tendance droite.

In [ ]:
cmd[["ca", "nart", "qte"]].corr().round(3)   ## toutes les paires

- `ca` et `qte` : **0,848**. Très lié.
- `ca` et `nart` : **0,382**. Nettement plus faible.

Que `qte` soit plus lié au montant que `nart` n'a rien de surprenant : le
montant se calcule à partir des **unités** vendues. Ce qui peut en revanche
vous surprendre, c'est que 0,382 soit aussi **bas** : une commande portant sur
40 produits distincts coûte en général plus cher qu'une commande qui en porte
3. Le lien existe, et ce coefficient le montre mal.

### Pearson et Spearman

La corrélation par défaut (**Pearson**) mesure l'alignement sur une droite.
Celle de **Spearman** raisonne sur les **rangs** : elle demande seulement si
l'un monte quand l'autre monte, quelle que soit la forme.

In [ ]:
print("Pearson  :", round(cmd["ca"].corr(cmd["nart"]), 3))   ## une droite
print("Spearman :", round(cmd["ca"].corr(cmd["nart"], method="spearman"), 3))

In [ ]:
cmd.plot(kind="scatter", x="nart", y="ca", alpha=0.3, figsize=(7, 4))
plt.title("ca en fonction de nart")
plt.xlabel("nart : nombre de produits distincts")
plt.ylabel("ca : montant de la commande (euros)")
plt.show()

**0,382 contre 0,671.** Même couple de variables, deux réponses très
différentes — et le nuage explique pourquoi : le lien existe, mais il est
**courbe**, et une poignée de commandes énormes étirent l'échelle. Pearson,
qui cherche une droite, sous-estime. Spearman, qui ne regarde que l'ordre,
voit le lien réel.

> 💡 **La règle.** Quand Pearson et Spearman divergent nettement, c'est que la
> relation n'est pas droite, ou que des valeurs extrêmes pèsent lourd. Les
> deux cas se voient sur le nuage, jamais dans le coefficient.

### Le poids des extrêmes

In [ ]:
seuil = cmd["ca"].quantile(0.99)   ## le montant des 1 % du haut
sans = cmd.query("ca < @seuil")    ## 20 lignes sur 1 955 en moins

print("avec :", round(cmd["ca"].corr(cmd["nart"]), 3))
print("sans :", round(sans["ca"].corr(sans["nart"]), 3),
      "-", len(cmd) - len(sans), "lignes en moins")

**0,382 → 0,489 en retirant 20 lignes sur 1 955.** Un coefficient cité sans le
nuage de points qui va avec n'est pas un résultat.

### Ce qu'un coefficient ne dit pas

**Vérification 1 — les deux variables sont-elles mesurées indépendamment ?**

`qte` et `ca` corrèlent à 0,848. Rien d'étonnant : **le montant se calcule à
partir des quantités**. Une variable et un de ses composants corrèlent toujours
fortement, et ce coefficient n'apprend rien sur le comportement d'achat — il
retrouve la formule de facturation. Une corrélation n'est une information que
si les deux variables sont mesurées **indépendamment** l'une de l'autre.

**Vérification 2 — le coefficient décrit-il une seule population ?**

In [ ]:
for pays in ["Belgique", "France", "Royaume-Uni", "Irlande"]:
    d = cmd.query("pays == @pays")   ## un marche a la fois
    print(f"{pays:<13}", round(d["ca"].corr(d["nart"]), 3))

**0,903 en Belgique, 0,234 en Irlande.** Le 0,382 global n'est le chiffre de
personne : c'est la moyenne de situations qui n'ont rien à voir entre elles.

Et l'Irlande, vous savez pourquoi : **deux clients**. Ce n'est pas le pays qui
produit ce comportement d'achat, c'est le **type de client** — deux grossistes
qui commandent en volume. Le pays n'est qu'une étiquette posée par-dessus.

Le type de client est ici une **variable de confusion** : une troisième
variable, non mesurée dans le fichier, qui influence à la fois ce qu'on croit
observer et la façon dont les observations se répartissent entre les groupes.

> ⚠️ **Le réflexe.** Avant de commenter un coefficient, demandez-vous **sur
> qui** il a été calculé, puis : **existe-t-il un Z qui expliquerait à la fois
> X et Y ?** Ici, Z est « ce client est un grossiste ».

C'est aussi, incidemment, la raison pour laquelle **une corrélation n'est pas
une causalité** : la formule qui produit le coefficient ne contient aucune
information sur ce qui cause quoi. Elle ne sait pas distinguer « X agit sur
Y » de « Z agit sur les deux ».

### La corrélation ne voit que les relations linéaires

Poussons le raisonnement jusqu'au bout. Voici deux colonnes où `y` est
**entièrement déterminé** par `x` — une parabole exacte, sans la moindre part
de hasard.

In [ ]:
x = pd.Series(range(-50, 51))
y = x ** 2   ## lien parfait, mais en forme de U

print("Pearson  :", round(x.corr(y), 3))   ## 0,000 : la droite n'y est pas
print("Spearman :", round(x.corr(y, method="spearman"), 3))

**0,000 des deux côtés.** Un lien parfait, et deux coefficients nuls.

La corrélation ne mesure pas « y a-t-il un lien ». Elle mesure « quand `x`
augmente, est-ce que `y` augmente **régulièrement** ? ». Ici la courbe descend
puis remonte, et les deux moitiés s'annulent exactement.

Notez que **Spearman ne sauve pas la mise**. Il corrige les relations courbes
mais **croissantes** — c'était le cas de `ca`/`nart`. Il ne peut rien faire
d'une relation qui change de sens en cours de route.

Ce cas n'a rien d'un exercice d'école : une remise fait monter les ventes puis
les fait baisser quand elle inquiète sur la qualité ; l'effet d'une pression
publicitaire plafonne puis se retourne.

> ⚠️ **Tracez toujours avant de conclure.** Un coefficient résume une forme ;
> il ne la montre pas, et il ne prévient jamais quand il ne convient pas.

## 3. Comparer deux groupes : hasard ou vrai écart ?

Le panier moyen français est-il différent du panier allemand ? Deux moyennes
ne sont jamais exactement égales : la question n'est pas « sont-elles
différentes ? » mais **« l'écart est-il assez grand pour ne pas être un
accident d'échantillonnage ? »**

Le **test t** répond exactement à ça :

> Si les deux marchés étaient identiques, à quelle fréquence observerait-on un
> écart au moins aussi grand que celui-ci ?

Cette fréquence, c'est la **p-value**.

In [ ]:
fr = cmd.query("pays == 'France'")["ca"]
de = cmd.query("pays == 'Allemagne'")["ca"]
print(len(fr), "vs", len(de), "| ecart :", round(fr.mean() - de.mean(), 2), "euros")

# equal_var=False : on ne suppose pas la meme dispersion des deux cotes
print("p-value :", round(stats.ttest_ind(fr, de, equal_var=False).pvalue, 3))

**p = 0,874.** Si les deux marchés étaient identiques, on observerait un écart
d'au moins 11 € dans **87 % des cas**. Cet écart n'a rien de remarquable.

La convention : on retient un écart quand **p < 0,05**.

| p-value | Ce qu'on en fait |
|---|---|
| p < 0,05 | l'écart serait rare sous l'hypothèse d'égalité : on le retient |
| p ≥ 0,05 | l'écart est compatible avec le hasard : **on ne conclut rien** |

> ⚠️ **La faute à ne pas commettre.** p = 0,874 ne dit **pas** que les deux
> marchés sont identiques. Il dit que ces données ne permettent pas de les
> départager. « Pas de différence détectable » et « pas de différence » sont
> deux affirmations différentes.

### Un cas où le test tranche — et le piège qui va avec

In [ ]:
uk = cmd.query("pays == 'Royaume-Uni'")["ca"]
irl = cmd.query("pays == 'Irlande'")["ca"]

print("p-value :", stats.ttest_ind(uk, irl, equal_var=False).pvalue)   ## minuscule

**p = 0,000000014.** Un écart comme celui-là ne s'explique pas par le hasard.

Avant d'écrire la recommandation, une question : **combien de personnes** y
a-t-il derrière ces commandes ?

In [ ]:
cmd.groupby("pays")["client_id"].nunique().nlargest(4)   ## des GENS

**Deux clients irlandais.** Contre 235 britanniques.

Le test t suppose que les observations sont **indépendantes** : que chaque
commande apporte une information nouvelle. Ici, 256 commandes viennent de deux
acheteurs. Ce ne sont pas 256 comportements d'achat, ce sont **deux**, répétés
128 fois chacun en moyenne. La p-value a été calculée comme s'il y en avait
256 : elle est beaucoup trop optimiste, et elle l'est dans le sens qui vous
plaît.

> ⚠️ **Avant tout test : comptez les individus, pas les lignes.** C'est le même
> réflexe `nunique` qu'en séance 2.2, et il aura la même importance quand vous
> constituerez un jeu d'apprentissage dans trois séances.

### Significatif ne veut pas dire important

In [ ]:
print("ecart :", round(irl.mean() - uk.mean(), 2), "euros")   ## la taille
print("rapport :", round(irl.mean() / uk.mean(), 2), "fois plus")

626 € d'écart, un panier 2,6 fois plus élevé : ici l'effet est énorme *et*
significatif.

Mais l'inverse arrive tout le temps : sur 100 000 commandes, un écart de 3 €
sort avec p < 0,001. Statistiquement indiscutable, commercialement sans
intérêt. **Affichez toujours l'écart dans son unité à côté de la p-value.**

---

Vous avez les quatre réflexes. La séance suivante s'en sert pour construire un
premier modèle qui **prédit** — et vous verrez qu'aucun d'entre eux ne devient
inutile.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| la moyenne, la médiane | `df["ca"].mean()`, `.median()` |
| tout d'un coup | `df["ca"].describe()` |
| un seuil (« les 10 % du haut ») | `df["ca"].quantile(0.9)` |
| la dispersion | `df["ca"].std()`, et `q3 - q1` pour la version robuste |
| découper en tranches | `pd.cut(df["ca"], bins=[...], labels=[...])` |
| la forme de la distribution | `df["ca"].plot(kind="hist", bins=40)` |
| voir un lien | `df.plot(kind="scatter", x="qte", y="ca", alpha=0.3)` |
| mesurer un lien | `df["ca"].corr(df["qte"])` |
| un lien courbe mais croissant | `.corr(..., method="spearman")` |
| toutes les paires d'un coup | `df[["ca", "nart", "qte"]].corr()` |
| comparer deux groupes | `stats.ttest_ind(a, b, equal_var=False).pvalue` |
| compter les individus | `df["client_id"].nunique()` |

## Les quatre réflexes à emporter dans le machine learning

1. **Moyenne et médiane, jamais l'une sans l'autre.** Presque tout ce qui est
   en euros est asymétrique. Un écart entre les deux vous dit qu'une poignée
   d'observations tire le résultat.

2. **Tracez avant de calculer.** Un coefficient résume une forme, il ne la
   montre pas — et il ne prévient jamais quand il ne convient pas.

3. **Comptez les individus, pas les lignes.** 256 commandes passées par deux
   acheteurs, ce sont deux comportements, pas 256. C'est vrai pour un test
   ici, et ce sera vrai pour un jeu d'apprentissage dans trois séances.

4. **« Significatif » ne veut pas dire « important ».** Affichez toujours la
   taille de l'écart, dans son unité, à côté de la p-value.

> Ces quatre réflexes ne sont pas des statistiques pour les statistiques :
> ce sont exactement les réflexes qui font la différence entre un modèle
> prédictif qu'on peut défendre et un modèle qu'on subit.